<a href="https://colab.research.google.com/github/omnia522006/CS50-Scratch-Key-game/blob/main/HousePredict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


train_path = '/content/sample_data/california_housing_train.csv'
test_path = '/content/sample_data/california_housing_test.csv'

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

y = train_data['median_house_value']
features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
            'total_bedrooms', 'population', 'households', 'median_income']
X = train_data[features]
test_X = test_data[features]


imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=features)
test_X = pd.DataFrame(imputer.transform(test_X), columns=features)


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_X_scaled = scaler.transform(test_X)
joblib.dump(scaler, 'scaler.pkl')


train_X, val_X, train_y, val_y = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}


random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='neg_mean_absolute_error',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(train_X, train_y)


best_rf_model = random_search.best_estimator_
print("Best hyperparameters:", random_search.best_params_)


val_preds = best_rf_model.predict(val_X)
print("Random Forest MAE:", mean_absolute_error(val_y, val_preds))
print("Random Forest RMSE:", np.sqrt(mean_squared_error(val_y, val_preds)))
print("Random Forest R2:", r2_score(val_y, val_preds))

best_rf_model.fit(X_scaled, y)

test_preds = best_rf_model.predict(test_X_scaled)
output = pd.DataFrame({'Id': test_data.index, 'SalePrice': test_preds})
output.to_csv('submission.csv', index=False)
print("Submission file created!")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best hyperparameters: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 30, 'bootstrap': True}
Random Forest MAE: 32134.54373886943
Random Forest RMSE: 49243.94071514473
Random Forest R2: 0.8240080982335781
Submission file created!


In [ ]:
import joblib
joblib.dump(best_rf_model, 'house_price_model.pkl')
loaded_model = joblib.load('house_price_model.pkl')
preds = loaded_model.predict(test_X_scaled)


In [ ]:
import joblib
joblib.dump(best_rf_model, 'house_price_model.pkl')
print("Model saved successfully!")
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved successfully!")


Model saved successfully!
Scaler saved successfully!


In [ ]:
from google.colab import files

files.download('house_price_model.pkl')

files.download('scaler.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>